# IEEE 118-bus DP3Ph / EMT3Ph performance comparison

## Main result

For a **1-second simulation**, replacing the conventional network branches, loads and transformers by their SSN formulations reduced the recorded calculation time from **3.439 s to 0.164 s in DP3Ph (21.0×)**. In EMT3Ph it fell from **6.684 s to 0.353 s (18.9×)**.

That headline comparison keeps the conventional fourth-order VBR synchronous generators in both runs. A separate, newer run replaces those machines with the new fourth-order SSN synchronous generator as well. The complete SSN case takes **0.694 s in DP** and **0.803 s in EMT** for the same 1-second simulated interval.

> **How to read the 21× result:** It is the ratio of the calculation times recorded for one short run of the conventional case and one run with SSN PiLines, SSN impedance loads and SSN transformers. The generators remain the same conventional VBR SG4 model in both. It is a strong measured result for this setup; the recorded comparison is not a multi-run statistical average.

This notebook summarizes the runs already present in the repository logs. It does not rerun the simulations.

## What was simulated

The dynamic cases are built from the MATPOWER IEEE case118 data in `dpsim/examples/cxx/Grids/IEEE118_Data.h` and the shared C++ builder in `dpsim/examples/cxx/Grids/IEEE118_Ph3.cpp`.

| Setting | DP3Ph | EMT3Ph |
|---|---:|---:|
| Network | IEEE 118-bus, three-phase | Same IEEE 118-bus, three-phase topology |
| Buses / synchronous generators / branches | 118 / 54 / 186 | Same |
| Branch breakdown | 175 pi-line branches + 11 transformer branches | Same |
| Nominal frequency | 60 Hz | 60 Hz |
| Time step | 1 ms | 0.5 ms |
| Steps in the short timing run | 1,000 | 2,000 |
| Simulated interval in the timing run | 1 s | 1 s |
| Solver | MNA with KLU | MNA with KLU |
| Initialization | Shared single-phase Newton-Raphson power flow, then domain initialization from PF | Same |

Loads, shunts and branch parameters come from the same case118 bus/branch data. All timings below are the simulator's logged **simulation calculation time**; they are not real-time factors and do not include an independently measured wall-clock total for building the case or solving the initial power flow. The regular logger is enabled in these short-run records (six bus voltages in DP; EMT logs the same buses with its configured downsampling and additional generator diagnostics). Logging configuration is held constant within each domain's set of variants.

The conventional generator is the fourth-order voltage-behind-reactance (VBR) SG4 model: `DP::Ph3::SynchronGenerator4OrderVBR` or `EMT::Ph3::SynchronGenerator4OrderVBR`. In the complete SSN variant, each generator instead uses the new domain-specific SSN SG4 component. Its initial P/Q is taken from the solved power flow, with the case bus voltage magnitude and angle used to initialize the machine state.

## The configurations

The table is an incremental component comparison. “SSN network” means all three passive network groups (pi lines, impedance loads, transformers) use SSN components while generators are still conventional VBR SG4. “All SSN” adds the SSN SG4 machines.

| Variant | Lines | Loads and shunts | Transformers | Generators |
|---|---|---|---|---|
| Conventional | Standard pi-line | Standard R/L/C loads in DP; RX loads in EMT | Standard transformer | Conventional VBR SG4 |
| SSN lines | SSN PiLine | Conventional | Conventional | Conventional VBR SG4 |
| SSN loads | Conventional | SSN impedance load | Conventional | Conventional VBR SG4 |
| SSN transformers | Conventional | Conventional | SSN transformer | Conventional VBR SG4 |
| SSN lines + loads | SSN PiLine | SSN impedance load | Conventional | Conventional VBR SG4 |
| SSN network (headline) | SSN PiLine | SSN impedance load | SSN transformer | Conventional VBR SG4 |
| SSN generators only | Conventional | Conventional | Conventional | SSN SG4 |
| All SSN | SSN PiLine | SSN impedance load | SSN transformer | SSN SG4 |

The isolated and partial substitutions help show the measured trend, but timings of individual rows are sensitive to runtime noise and interactions between component/task structure. The strongest controlled comparisons here are the two like-for-like endpoint pairs within each domain.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Values read from the matching DPsim log files in logs/.
# A row exists only where the corresponding run was recorded.
records = [
    ("DP3Ph", "Conventional", 3.439234, "VBR SG4; standard lines, loads, transformers"),
    (
        "DP3Ph",
        "SSN PiLine only",
        1.321257,
        "VBR SG4; other network components conventional",
    ),
    (
        "DP3Ph",
        "SSN impedance load only",
        3.980360,
        "VBR SG4; other network components conventional",
    ),
    (
        "DP3Ph",
        "SSN transformer only",
        3.822380,
        "VBR SG4; other network components conventional",
    ),
    ("DP3Ph", "SSN lines + loads", 2.142486, "VBR SG4; conventional transformers"),
    (
        "DP3Ph",
        "SSN network (headline)",
        0.163708,
        "VBR SG4; SSN lines, loads, transformers",
    ),
    (
        "DP3Ph",
        "SSN generator only",
        4.160715,
        "SSN SG4; conventional network components",
    ),
    ("DP3Ph", "All SSN", 0.694081, "SSN SG4, lines, loads, transformers"),
    (
        "EMT3Ph",
        "Conventional",
        6.684139,
        "VBR SG4; standard lines, loads, transformers",
    ),
    (
        "EMT3Ph",
        "SSN PiLine only",
        1.565281,
        "VBR SG4; other network components conventional",
    ),
    ("EMT3Ph", "SSN lines + loads", 4.047447, "VBR SG4; conventional transformers"),
    (
        "EMT3Ph",
        "SSN network (headline)",
        0.353131,
        "VBR SG4; SSN lines, loads, transformers",
    ),
    ("EMT3Ph", "All SSN", 0.802932, "SSN SG4, lines, loads, transformers"),
]
times = pd.DataFrame(
    records, columns=["Domain", "Variant", "Calculation time (s)", "Component setup"]
)
baseline = times[times["Variant"] == "Conventional"].set_index("Domain")[
    "Calculation time (s)"
]
times["Speedup vs conventional"] = times.apply(
    lambda r: baseline[r["Domain"]] / r["Calculation time (s)"], axis=1
)
display(times.round({"Calculation time (s)": 3, "Speedup vs conventional": 2}))

## Headline comparison

In [ ]:
headline = times[
    times["Variant"].isin(["Conventional", "SSN network (headline)", "All SSN"])
].copy()
order = ["Conventional", "SSN network (headline)", "All SSN"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.7), sharey=False)
colors = {
    "Conventional": "#6b7280",
    "SSN network (headline)": "#2563eb",
    "All SSN": "#0f9d78",
}
for ax, domain in zip(axes, ["DP3Ph", "EMT3Ph"]):
    sub = headline[headline["Domain"] == domain].set_index("Variant").loc[order]
    bars = ax.barh(order, sub["Calculation time (s)"], color=[colors[x] for x in order])
    ax.invert_yaxis()
    ax.set_title(domain)
    ax.set_xlabel("Calculation time for 1 s simulated (s)")
    ax.grid(axis="x", alpha=0.25)
    for bar, (_, row) in zip(bars, sub.iterrows()):
        ax.text(
            bar.get_width() + max(sub["Calculation time (s)"]) * 0.025,
            bar.get_y() + bar.get_height() / 2,
            f'{row["Calculation time (s)"]:.3f} s  ({row["Speedup vs conventional"]:.1f}×)',
            va="center",
            fontsize=9,
        )
fig.suptitle(
    "The network-only SSN case is the source of the ~21× DP result", y=1.03, fontsize=13
)
fig.tight_layout()
plt.show()

for domain in ["DP3Ph", "EMT3Ph"]:
    t0 = float(baseline[domain])
    tnet = float(
        times[(times.Domain == domain) & (times.Variant == "SSN network (headline)")][
            "Calculation time (s)"
        ].iloc[0]
    )
    tall = float(
        times[(times.Domain == domain) & (times.Variant == "All SSN")][
            "Calculation time (s)"
        ].iloc[0]
    )
    print(
        f"{domain}: conventional / SSN network = {t0/tnet:.2f}×; conventional / all-SSN = {t0/tall:.2f}×"
    )

### What the 21× number does and does not mean

For DP3Ph, the logged calculation time drops from **3.439234 s** to **0.163708 s** when the 175 pi-line branches, loads/shunts and 11 transformers use their SSN formulations:

\[
\text{speedup} = \frac{3.439234}{0.163708} = 21.01\times.
\]

Both runs simulate 1 second at 1 ms (1,000 time steps), use the same IEEE118 operating point, solver and conventional VBR fourth-order synchronous generators. The EMT endpoint comparison is **6.684139 / 0.353131 = 18.93×**, with both EMT runs at 0.5 ms (2,000 steps).

The **all-SSN** run is a different and more demanding generator comparison: it also replaces all 54 conventional VBR machines with the new SSN SG4. Its speedups vs conventional are **4.95× in DP** and **8.32× in EMT**. This is still faster than the conventional endpoint, but is not the 21× configuration. In the 1-second logs, replacing VBR SG4 by SSN SG4 while keeping the passive network SSN increases time from 0.164 s to 0.694 s in DP and from 0.353 s to 0.803 s in EMT.

## Full 30-second run: runtime and numerical behavior

The complete SSN setup (SSN lines, loads, transformers and fourth-order generators) was also run for 30 seconds. These are separate validation runs, not the 1-second timing records above.

| Domain | Time step | Steps | Logged calculation time | End state |
|---|---:|---:|---:|---|
| DP3Ph, complete SSN | 1 ms | 30,000 | 21.09 s | Voltage traces remained bounded; over the six monitored buses, the sampled average-magnitude difference from the existing reference was about 8–235 V. |
| EMT3Ph, complete SSN | 0.5 ms | 60,000 | 18.55 s | Bounded, but some bus RMS voltages shifted by roughly 3–4% vs conventional. Gen10 speed drifted from 1.000 to about 0.977 pu. |

The EMT case therefore ran to completion without unbounded voltages, but its electromechanical response does not yet match the VBR reference as tightly as DP. The SSN generator implementation needs further accuracy work before presenting EMT as a validated equivalent model. The 21× timing headline comes from the passive-network SSN comparison with conventional generators and should not be used to imply the new SSN generator itself is faster.

## Provenance and reproducibility

The short-run values above are transcribed from `logs/IEEE118_DP3Ph_CPP*/IEEE118_*.log` and `logs/IEEE118_EMT3Ph_CPP*/IEEE118_*.log`, using each log's `Simulation calculation time` field. The C++ setup and run switches are in `dpsim/examples/cxx/Grids/IEEE118_Ph3.cpp`; the case data are in `dpsim/examples/cxx/Grids/IEEE118_Data.h`.

Timing caveats:

- These are **single logged runs**, not means across repeated trials. Small differences between incremental variants can reflect machine load and component/task interactions.
- Each DP row is comparable with another DP row; each EMT row with another EMT row. DP and EMT use different fixed time steps, so their raw seconds should not be interpreted as a solver-only contest at equal step count.
- Solver, timestep, simulated duration, initialization route and logger policy are kept fixed within each domain's comparison. The setup uses MNA with KLU.
- Simulation calculation time is the simulator's reported calculation interval, not a separately benchmarked end-to-end process time. The initial power flow is performed separately.
- The 21× result is specifically for converting the **passive network components** to SSN while retaining VBR SG4 machines. The all-SSN SG4 result is reported separately above.

For a publication-grade benchmark, rerun each endpoint several times on the same otherwise idle machine, report median and spread, and keep the executable/build and logger settings fixed.